# Primer XGBoost: conteos y alerta semanal

Este notebook ejecuta `src.modeling.train`. Usa el gold validado para h=2 y h=4, compara cuatro frente a seis variables sobre las mismas filas y entrena regresión de `log1p(casos_Dengue)` y clasificación directa de `brote`. La regresión produce también una alerta al aplicar el umbral histórico de la fila. La persistencia es la referencia.

Validación: temporadas 2021–2023. Prueba principal: temporada 2024. Prueba adicional del cambio de fuente: año calendario 2025 completo. Se reentrena antes del primer origen de cada temporada y se fija el umbral de probabilidad solo con la validación.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.modeling.train import ejecutar_experimento

reporte = ejecutar_experimento()

In [ ]:
import pandas as pd

filas = []
for h, bloque in reporte["horizontes"].items():
    for variante, experimento in bloque["variantes"].items():
        for fold in experimento["folds"]:
            if not fold["tipo"].startswith("prueba"):
                continue
            for enfoque, valores in (("regresion", fold["regresion"]["alerta_derivada"]),
                                      ("clasificacion", fold["clasificacion"]["alerta_directa"]),
                                      ("persistencia", fold["persistencia"]["alerta_derivada"])):
                filas.append({"h": int(h), "variante": variante, "bloque": fold["bloque"],
                              "enfoque": enfoque, "positivos": valores["positivos"],
                              "VP": valores["vp"], "FP": valores["fp"],
                              "precision": valores["precision"], "recall": valores["recall"],
                              "F1": valores["f1"], "AUPRC": valores["auprc"]})
resumen = pd.DataFrame(filas)
display(resumen.sort_values(["h", "bloque", "variante", "enfoque"]))

El año calendario 2025 tiene solo tres positivos, por lo que sus métricas de alerta son muy inestables y se reportan como sensibilidad separada. No escoger el enfoque con las métricas de prueba: la decisión operativa sobre falsos avisos y casos perdidos sigue pendiente de discusión. Los modelos, predicciones y métricas quedan en `models/experimentos/` y `docs/modeling/metricas/`.